# STT 10 — Whisper contre notre modèle, sur le test officiel

**But :** savoir à quelle distance notre modèle (WER test 43,3 %) est du **Whisper utilisé par le robot**, sur **les mêmes 15 936 clips**, avec les mêmes règles de comptage, et à **vitesse mesurée sur le même GPU**.

**Décisions (journal du 27/11) :**
1. **Whisper comme sur le robot** (`jone/stt.py`, `config.yaml`) : large-v3-turbo, `faster-whisper`, `int8_float16`, `beam_size=1`, mais **langue forcée en français** (notre modèle ne connaît que le français : on compare la reconnaissance). Sur 1 000 clips tirés au hasard, on compte aussi les erreurs de **détection automatique** de langue (réglage réel du robot).
2. **Deux façons de compter**, pour les deux modèles : **(a)** notre protocole (`normalize()`, ponctuation gardée : celui des 43,3 %) et **(b)** sans ponctuation (`. , ? ! -` retirés), le protocole standard des publications sur Whisper.
3. **Le test officiel entier.** On compte les clips où Whisper écrit des **chiffres** (« 3 » au lieu de « trois ») : ils le pénalisent un peu.
4. **RTF sur le même GPU T4**, un clip à la fois, pour les deux modèles.

**Lancement :** envoyé par la ligne de commande `kaggle`, comme nouvelle version du notebook de `stt_08` (GPU T4, Internet, Secret `MDC_API_KEY` et `best.pt` de `notebook3bfc7d7729` déjà attachés).

**Règles de données (conditions MDC) :** les données vont dans `/tmp`. **Aucune phrase n'est affichée ni sauvegardée**, ni la nôtre ni celle de Whisper : seulement des chiffres, dans `whisper_result.json`.

**Précaution :** OpenAI ne publie pas la liste exacte des données de Whisper ; on ne peut pas garantir qu'il n'a jamais entendu ces clips.

## Cellule 1 — Code, machine, et Whisper chargé tout de suite

Whisper est chargé **avant** le téléchargement de 18 min : si `faster-whisper` ne trouve pas le GPU, on le sait en 2 minutes, pas après 20.

- `faster-whisper` utilise sa propre bibliothèque (CTranslate2), qui a besoin de **cuDNN**. On pré-charge les fichiers cuDNN fournis avec PyTorch, sinon elle peut ne pas les trouver.
- `WhisperSTT` (code de l'équipe) **repasse en silence sur CPU** si le GPU échoue. Sur CPU, 15 936 clips prendraient des heures : on **vérifie** donc que le modèle est bien sur `cuda`, et on arrête sinon.

In [ ]:
import time, sys, os, glob, json, ctypes
SESSION_START = time.time()

!git clone -b stt-transformer-junior --depth 1 https://github.com/mohamed-943/projet-J-ONE.git
%cd projet-J-ONE/recherche/stt_transformer
!pip install -q datacollective soundfile faster-whisper
!nvidia-smi --query-gpu=name,memory.total --format=csv
sys.path.insert(0, "/kaggle/working/projet-J-ONE")          # pour importer jone.stt (le code du robot)

# cuDNN fourni avec PyTorch : pré-chargé pour que CTranslate2 (faster-whisper) le trouve
for lib in sorted(glob.glob("/usr/local/lib/python3*/*-packages/nvidia/cudnn/lib/libcudnn*.so*")):
    try:
        ctypes.CDLL(lib, mode=ctypes.RTLD_GLOBAL)
    except OSError:
        pass

import numpy as np
from jone.stt import WhisperSTT

# Exactement le réglage du robot (config.yaml), sauf la langue forcée en français (décision 1)
stt = WhisperSTT(model="large-v3-turbo", device="cuda", compute_type="int8_float16",
                 beam_size=1, language="fr", languages=("fr",))
assert stt.model.model.device == "cuda", "Whisper est sur CPU : arrêt (beaucoup trop lent pour 15 936 clips)"
stt.transcribe(np.zeros(16000, dtype=np.float32))           # échauffement : 1 s de silence
print(f"Whisper prêt sur GPU après {(time.time() - SESSION_START) / 60:.1f} min", flush=True)

## Cellule 2 — Téléchargement et décompression (environ 18 min)

Exactement comme dans `stt_08`.

In [ ]:
import os, glob, json, subprocess
from kaggle_secrets import UserSecretsClient

# La clé est lue dans le Secret Kaggle : elle n'apparaît jamais dans le code.
secrets = UserSecretsClient()
try:
    os.environ["MDC_API_KEY"] = secrets.get_secret("MDC_API_KEY")
except Exception:
    os.environ["MDC_API_KEY"] = secrets.get_secret("mdc_api_key")
from datacollective import download_dataset

BIG_ID = "cmu5kub4600pkmh078n7n5ogl"           # Scripted Speech 27.0 FR
BIG_DIR = "/tmp/big"                           # /tmp : jamais sauvegardé avec le notebook (conditions MDC)

download_dataset(BIG_ID, download_directory=BIG_DIR)
archive = glob.glob(f"{BIG_DIR}/*.tar.gz")[0]
subprocess.run(["tar", "-xzf", archive, "-C", BIG_DIR], check=True)
os.remove(archive)                             # libère les 30 Go de l'archive
BASE = os.path.dirname(glob.glob(f"{BIG_DIR}/**/train.tsv", recursive=True)[0])
print(f"Données prêtes après {(time.time() - SESSION_START) / 60:.1f} min de session")

## Cellule 3 — Le test officiel

Comme dans `stt_08` : `load_split` lit `test.tsv`, normalise le texte et retire les phrases avec des caractères hors vocabulaire (271 clips). On garde ainsi **les mêmes 15 936 clips** que pour le 43,3 %.

In [ ]:
import pandas as pd
import torch
from stt.dataset import load_split

test_table = load_split([BASE], "test")
print(f"Test officiel : {len(test_table):,} clips gardés (attendu : 15,936)", flush=True)

## Cellule 4 — Notre meilleur modèle

Comme dans `stt_08` : le `best.pt` qui a le plus petit WER dev (époque 19, 37,38 %).

In [ ]:
from stt.model import SpeechTransformer

device = "cuda"
found = sorted(glob.glob("/kaggle/input/**/best.pt", recursive=True))
assert found, "Aucun best.pt : ajoute la sortie du notebook d'entraînement avec Add Input"
checkpoints = [(torch.load(p, map_location=device, weights_only=False), p) for p in found]
ckpt, path = min(checkpoints, key=lambda c: c[0]["wer"])     # le plus petit WER dev
del checkpoints

model = SpeechTransformer().to(device)                       # taille S, comme à l'entraînement
model.load_state_dict(ckpt["model"])
model.eval()                                                 # dropout coupé, aucun apprentissage
print(f"Modèle chargé : step {ckpt['step']:,} | WER dev {ckpt['wer']:.2f} %", flush=True)

## Cellule 5 — La boucle commune : chaque clip passe dans les deux modèles (environ 50 min)

Pour **chaque clip**, une seule lecture de l'audio, puis :
1. **notre modèle**, un clip à la fois (batch de 1), chronométré avec `torch.cuda.synchronize()` (le GPU calcule en arrière-plan : sans lui, le chronomètre s'arrêterait trop tôt) ;
2. **Whisper**, chronométré par le code de l'équipe (`elapsed_ms`) ;
3. les erreurs de mots et de caractères des deux modèles, avec les deux protocoles (a) et (b).

Seuls des **nombres** sont gardés. Un point d'étape est affiché tous les 2 000 clips.

Un `DataLoader` avec 4 *workers* lit les mp3 en avance pendant que le GPU calcule (`batch_size=None` : un clip à la fois).

In [ ]:
import re
from torch.utils.data import Dataset, DataLoader
from stt.audio import load_audio, log_mel, TARGET_SR
from stt.metrics import greedy_decode, edit_distance
from stt.text import normalize

class Clips(Dataset):
    # Renvoie l'audio d'un clip et son numéro (le texte reste dans test_table)
    def __len__(self):
        return len(test_table)
    def __getitem__(self, i):
        return load_audio(test_table["audio_path"][i]), i

def no_punct(text):
    # Protocole (b) : ponctuation retirée, comme dans les publications sur Whisper
    return re.sub(r" +", " ", re.sub(r"[.,?!-]", " ", text)).strip()

PROTOCOLS = {"a": lambda t: t, "b": no_punct}
rows, t_loop = [], time.time()

with torch.no_grad():
    for waveform, i in DataLoader(Clips(), batch_size=None, shuffle=False, num_workers=4):
        ref = test_table["text"][i]

        # 1. Notre modèle, batch de 1
        torch.cuda.synchronize(); t0 = time.perf_counter()
        features = log_mel(waveform).unsqueeze(0).to(device)
        log_probs, out_lengths = model(features, torch.tensor([features.shape[1]], device=device))
        hyp_ours = greedy_decode(log_probs, out_lengths)[0]
        torch.cuda.synchronize(); ours_s = time.perf_counter() - t0

        # 2. Whisper, comme sur le robot (langue forcée en français)
        result = stt.transcribe(waveform.numpy())
        hyp_whisper = normalize(result.text)                 # même normalisation que nos références

        # 3. Uniquement des nombres (ref, hyp_ours, hyp_whisper ne sont jamais affichés ni sauvegardés)
        row = {"seconds": len(waveform) / TARGET_SR, "ours_s": ours_s, "whisper_s": result.elapsed_ms / 1000,
               "whisper_digits": bool(re.search(r"\d", hyp_whisper))}
        for name, hyp in (("ours", hyp_ours), ("whisper", hyp_whisper)):
            for p, f in PROTOCOLS.items():
                r, h = f(ref), f(hyp)
                row[f"{name}_{p}_we"] = edit_distance(r.split(), h.split()); row[f"{name}_{p}_w"] = len(r.split())
                row[f"{name}_{p}_ce"] = edit_distance(list(r), list(h));     row[f"{name}_{p}_c"] = len(r)
        rows.append(row)

        if len(rows) % 2000 == 0:                            # point d'étape (protocole a)
            d = pd.DataFrame(rows)
            print(f"{len(rows):>6,} clips | {(time.time() - t_loop) / 60:5.1f} min | "
                  f"WER nous {100 * d.ours_a_we.sum() / d.ours_a_w.sum():.1f} % | "
                  f"Whisper {100 * d.whisper_a_we.sum() / d.whisper_a_w.sum():.1f} %", flush=True)

res = pd.DataFrame(rows)
print(f"{len(res):,} clips évalués en {(time.time() - t_loop) / 60:.1f} min", flush=True)

## Cellule 6 — Détection automatique de la langue (1 000 clips au hasard)

Le robot ne force pas la langue : il laisse Whisper choisir entre français et anglais. Sur 1 000 clips tirés au hasard (graine fixe, résultat reproductible), on compte combien de fois Whisper **ne répondrait pas en français**. C'est une erreur que notre modèle, français seulement, ne peut pas faire.

In [ ]:
sample = np.random.default_rng(0).choice(len(test_table), size=1000, replace=False)
stt.language, stt.languages = None, ["fr", "en"]            # réglage réel du robot (config.yaml)
wrong_language = 0
for i in sample:
    result = stt.transcribe(load_audio(test_table["audio_path"][int(i)]).numpy())
    wrong_language += (result.language != "fr")              # le texte n'est pas gardé
stt.language, stt.languages = "fr", ["fr"]                  # on remet le réglage de la cellule 1
print(f"Langue mal détectée : {wrong_language} clips sur 1 000 ({wrong_language / 10:.1f} %)", flush=True)

## Cellule 7 — Les résultats

- **WER et CER** des deux modèles, avec les deux protocoles, globaux et par durée de clip ;
- **RTF** (temps total ÷ durée totale) et **latence médiane** sur le même GPU, sans les 5 premiers clips (échauffement) ;
- part des clips où Whisper écrit des chiffres.

Tout est écrit dans `whisper_result.json` (*Output*) : **uniquement des chiffres**.

In [ ]:
def rates(df, name, p):
    return {"wer": round(100 * df[f"{name}_{p}_we"].sum() / df[f"{name}_{p}_w"].sum(), 2),
            "cer": round(100 * df[f"{name}_{p}_ce"].sum() / df[f"{name}_{p}_c"].sum(), 2)}

summary = {"checkpoint": {"path": path, "step": ckpt["step"], "dev_wer": round(ckpt["wer"], 2)},
           "whisper": {"model": "large-v3-turbo", "compute_type": "int8_float16", "beam_size": 1, "language": "fr"},
           "clips": len(res), "overall": {}, "by_duration": {}}
print(f"TEST OFFICIEL ({len(res):,} clips)")
for name in ("ours", "whisper"):
    for p in PROTOCOLS:
        r = summary["overall"][f"{name}_{p}"] = rates(res, name, p)
        print(f"  {name:8s} protocole ({p}) | WER {r['wer']:6.2f} % | CER {r['cer']:6.2f} %")

bins, labels = [0, 3, 5, 7, float("inf")], ["< 3 s", "3-5 s", "5-7 s", ">= 7 s"]
res["tranche"] = pd.cut(res["seconds"], bins=bins, labels=labels, right=False)
print("\nPar durée de clip (protocole b) :")
for label in labels:
    part = res[res["tranche"] == label]
    if len(part):
        o, w = rates(part, "ours", "b"), rates(part, "whisper", "b")
        summary["by_duration"][label] = {"clips": len(part), "ours_b": o, "whisper_b": w}
        print(f"  {label:>7} | {len(part):>6,} clips | WER nous {o['wer']:6.2f} % | Whisper {w['wer']:6.2f} %")

timed = res.iloc[5:]                                        # 5 clips d'échauffement non comptés
summary["speed_t4"] = {name: {"rtf": round(timed[f"{name}_s"].sum() / timed["seconds"].sum(), 4),
                              "latence_mediane_ms": round(1000 * timed[f"{name}_s"].median(), 1)}
                       for name in ("ours", "whisper")}
summary["whisper_digits_pct"] = round(100 * res["whisper_digits"].mean(), 2)
summary["wrong_language_auto_pct"] = wrong_language / 10
print("\nVitesse sur T4 (batch de 1) :")
for name, s in summary["speed_t4"].items():
    print(f"  {name:8s} | RTF {s['rtf']:.4f} | latence médiane {s['latence_mediane_ms']:.0f} ms")
print(f"\nClips où Whisper écrit des chiffres : {summary['whisper_digits_pct']:.2f} %")
print(f"Langue mal détectée en mode automatique : {summary['wrong_language_auto_pct']:.1f} %")

with open("/kaggle/working/whisper_result.json", "w") as f:
    json.dump(summary, f, indent=1)
print("\nwhisper_result.json écrit dans Output (aucune phrase, uniquement des chiffres)")
print(f"Session totale : {(time.time() - SESSION_START) / 60:.1f} min")